In [1]:
import os
import sys

os.chdir("../")

In [2]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

d:\Python\Med chat bot\med\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Extract Text from PDF

def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    document=loader.load()
    return document

In [4]:
extracted_data = load_pdf_files("data")

In [5]:
len(extracted_data)

637

In [6]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [7]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [8]:
#split the document into chunks

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        
    )
    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk

texts_chunk = text_split(minimal_docs)   

In [9]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\amith\AppData\Local\Temp\ipykernel_2220\30168171.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [10]:
vector = embedding.embed_query("Hello world")
vector

[-0.03447723761200905,
 0.031023172661662102,
 0.006734994240105152,
 0.026108955964446068,
 -0.03936205431818962,
 -0.16030253469944,
 0.06692399829626083,
 -0.006441488862037659,
 -0.04745050147175789,
 0.014758973382413387,
 0.07087533175945282,
 0.05552758648991585,
 0.019193360581994057,
 -0.02625136263668537,
 -0.010109484195709229,
 -0.026940476149320602,
 0.022307496517896652,
 -0.022226696833968163,
 -0.14969263970851898,
 -0.0174931101500988,
 0.007676286157220602,
 0.05435218662023544,
 0.003254459472373128,
 0.0317259356379509,
 -0.08462143689393997,
 -0.029405932873487473,
 0.05159565806388855,
 0.04812400043010712,
 -0.0033148087095469236,
 -0.058279216289520264,
 0.041969332844018936,
 0.022210735827684402,
 0.12818878889083862,
 -0.022338945418596268,
 -0.01165622379630804,
 0.06292835623025894,
 -0.03287626802921295,
 -0.09122609347105026,
 -0.031175386160612106,
 0.052699603140354156,
 0.047034852206707,
 -0.0842030793428421,
 -0.03005618415772915,
 -0.020744716748595

In [11]:
print("Vectors:", len(vector))

Vectors: 384


In [12]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [13]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["GROQ_API_KEY"]=GROQ_API_KEY

In [14]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [15]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [17]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

# ADD EXTRA DOCUMENTS


In [18]:
dswith= Document(
    page_content="Hello, world!", 
    metadata={"source": "https://example.com"}
)

In [19]:
docsearch.add_documents(documents=[dswith])

['504a2c74-b41f-4976-8ce3-b9eb3ed5907b']

In [20]:
retriever = docsearch.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 5}
)

In [21]:
retriever_doc = retriever.invoke("What is Acne?")
retriever_doc

[Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='KEY TERMS\nAcne—A skin condition in which raised bumps,\npimples, and cysts form on the face, neck, shoul-\nders and upper back.\nBacteria—Tiny, one-celled forms of life that cause\nmany diseases and infections.\nBowel—The intestine; a tube-like structure that\nextends from the stomach to the anus. Some diges-\ntive processes are carried out in the bowel before\nfood passes out of the body as waste.\nCyst—An abnormal sac or enclosed cavity in the\nbody, filled with liquid or partially solid material.'),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. 

In [32]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY,
    
)
# response = llm.invoke("Explain Quantum Computing.")
# print(response.content)

In [33]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [34]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [36]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [37]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of a chemical from the pituitary gland in the brain, leading to increased growth in bone and soft tissue. This is usually due to an excess of growth hormone (GH) in the body. When this abnormality occurs in children, it's called gigantism, resulting in exceptional growth of long bones due to open bony growth plates.
